In [1]:
pip install pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 5.1 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=fab3bfd8a95329a336768dca79f98b13829397070a40a42c2e2357a7d2f1e800
  Stor

In [2]:
pip install matgl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.9/366.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 63.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

from pymatgen.core import Structure
import matgl
from matgl.ext.pymatgen import Structure2Graph

In [4]:
import torch
import torch.nn as nn
from torch_geometric.nn import CGConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim.lr_scheduler import StepLR
import numpy as np

class CGCNN(nn.Module):
    def __init__(self, node_features=8, edge_features=40,
                 hidden_dim=64, num_conv_layers=3, dropout=0.1):
        super(CGCNN, self).__init__()

        # Initial embedding of node features
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # Graph convolutional layers
        self.conv_layers = nn.ModuleList([
            CGConv(hidden_dim, dim=edge_features, batch_norm=True)
            for _ in range(num_conv_layers)
        ])

        self.dropout = nn.Dropout(dropout)

        # Readout MLP
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = (
            data.x, data.edge_index, data.edge_attr, data.batch
        )

        # Embed node features
        x = self.node_embedding(x)
        x = torch.relu(x)

        # Graph convolutions
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr)
            x = torch.relu(x)
            x = self.dropout(x)

        # Global pooling: aggregate all atoms into one vector
        x = global_mean_pool(x, batch)

        # Predict
        out = self.fc(x)
        return out.squeeze(-1)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)

        y = batch.y.view(-1)  # flatten to 1D

        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)

            # Use squeeze(-1) to only squeeze the last dimension
            y = batch.y.view(-1)  # flatten to 1D regardless of shape

            loss = criterion(pred, y)
            total_loss += loss.item()
            preds.extend(pred.cpu().numpy())
            targets.extend(y.cpu().numpy())

    mae = np.mean(np.abs(np.array(preds) - np.array(targets)))
    return total_loss / len(loader), mae


def train_cgcnn(graphs, test_size=0.2, val_size=0.1,
                hidden_dim=64, num_conv_layers=3,
                epochs=100, batch_size=32, lr=1e-3):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Split dataset
    n = len(graphs)
    n_test  = int(n * test_size)
    n_val   = int(n * val_size)
    n_train = n - n_test - n_val

    train_graphs = graphs[:n_train]
    val_graphs   = graphs[n_train:n_train + n_val]
    test_graphs  = graphs[n_train + n_val:]

    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_graphs,   batch_size=batch_size)
    test_loader  = DataLoader(test_graphs,  batch_size=batch_size)

    # Model, optimizer, loss
    node_features = graphs[0].x.shape[1]
    edge_features = graphs[0].edge_attr.shape[1]

    model = CGCNN(
        node_features=node_features,
        edge_features=edge_features,
        hidden_dim=hidden_dim,
        num_conv_layers=num_conv_layers
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = StepLR(optimizer, step_size=30, gamma=0.5)
    criterion = nn.MSELoss()

    # Training loop
    best_val_loss = float('inf')
    best_model_state = None
    loss = []
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_mae = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:03d} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"Val MAE: {val_mae:.4f}")
            loss.append([train_loss,val_loss])

    # Load best model and evaluate on test set
    model.load_state_dict(best_model_state)
    test_loss, test_mae = evaluate(model, test_loader, criterion, device)

    predictions = []
    targets = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            preds = model(batch)
            predictions.extend(preds.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())




    print(f"\nTest Loss: {test_loss:.4f} | Test MAE: {test_mae:.4f}")

    return model, test_mae, loss, predictions, targets

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Copy from Drive to local Colab storage
import shutil
shutil.copy('/content/drive/My Drive/Colab Notebooks/crystal_graphs_matminer_features.pt', '/content/crystal_graphs_matminer_features.pt')


Mounted at /content/drive


'/content/crystal_graphs_matminer_features.pt'

In [6]:
graphs = torch.load('/content/crystal_graphs_matminer_features.pt', weights_only=False)

filtered_data = [data for data in graphs if data.y.item() != 0]
# Train
model, test_mae, loss, predictions, targets = train_cgcnn(
    filtered_data,
    hidden_dim=64,
    num_conv_layers=3,
    epochs=300,
    batch_size=32,
    lr=1e-3,
    #dropout=0.1  # light regularization
)


Using device: cuda
Epoch 010 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 020 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 030 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 040 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 050 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 060 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 070 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 080 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 090 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 100 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 110 | Train Loss: nan | Val Loss: nan | Val MAE: nan
Epoch 120 | Train Loss: nan | Val Loss: nan | Val MAE: nan


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'cgcnn_model_matminer_features.pt')
np.savetxt('cgcnn_loss_hist_matminer_features.csv', loss)
np.savetxt('cgcnn_pred_matminer_features.csv', predictions)
np.savetxt('cgcnn_targets_matminer_features.csv', targets)

In [ ]:

ls

cgcnn_loss_hist.csv  cgcnn_pred.csv     crystal_graphs.pt  sample_data/
cgcnn_model.pt       cgcnn_targets.csv  drive/
